# 第 13 章 エンドツーエンドの実例

生データの前処理から、3 つのモデルの比較・評価までを 1 本のパイプラインに通します。

対応する記事: [第 13 章 エンドツーエンドの実例（Kotlin 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/kotlin/ch13.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch13.*

## 生データ

「40 歳未満かつ収入 400 超なら購入」という規則に従う擬似データを作ります。**9 行に 1 行は年齢が欠損** しています。実務のデータは、そのままではモデルに渡せません。

In [2]:
val random = kotlin.random.Random(1)
val cities = listOf("tokyo", "osaka", "kyoto")
val rows = List(40) { i ->
    val age = random.nextInt(18, 71)
    val income = random.nextInt(200, 901)
    mapOf(
        "age" to (if (i % 9 == 0) "" else age.toString()),
        "income" to income.toString(),
        "city" to cities[random.nextInt(cities.size)],
        "bought" to if (age < 40 && income > 400) "yes" else "no",
    )
}

rows.take(5).forEach { println(it) }

{age=, income=374, city=osaka, bought=no}
{age=65, income=683, city=kyoto, bought=no}


{age=66, income=516, city=kyoto, bought=no}
{age=43, income=711, city=osaka, bought=no}


{age=21, income=855, city=osaka, bought=yes}


## 前処理

3 つの処理を通します。それぞれ「やらないとどうなるか」が明確です。

| 処理 | やらないとどうなるか |
| :--- | :--- |
| 欠損補完（中央値） | 学習が落ちる。平均だと 1 件の外れ値が全欠損を汚染する |
| 正規化 | 値の大きい特徴量（収入）だけが効く |
| One-Hot | カテゴリに存在しない大小関係が生まれる |

In [3]:
val dataset = buildDataset(rows, "bought")

println("特徴量: " + dataset.featureNames)
println("陽性 ${dataset.labels.sum()} 件 / 全 ${dataset.labels.size} 件")
println()
dataset.points.zip(dataset.labels).take(5).forEach { (point, label) ->
    println(point.map { "%.3f".format(it) }.toString() + " → " + label)
}

特徴量: [age, income, city=kyoto, city=osaka, city=tokyo]
陽性 11 件 / 全 40 件



[0.654, 0.237, 0.000, 1.000, 0.000] → 0
[0.904, 0.686, 1.000, 0.000, 0.000] → 0


[0.923, 0.443, 1.000, 0.000, 0.000] → 0
[0.481, 0.727, 0.000, 1.000, 0.000] → 0


[0.058, 0.936, 0.000, 1.000, 0.000] → 1


## 欠損補完と正規化を個別に確かめる

**平均ではなく中央値を使うのは、外れ値に引きずられないため** です。`[1, 2, 3, 1000]` の平均は 251.5 ですが、中央値は 2.5 です。

In [4]:
println("欠損補完: " + imputeMissing(listOf(1.0, 2.0, 3.0, 1000.0, null)))
println("正規化  : " + normalize(listOf(10.0, 20.0, 30.0)))
println("定数列  : " + normalize(listOf(5.0, 5.0, 5.0)) + " ← 0 除算しない")
println("One-Hot : " + oneHot(listOf("b", "a", "b")))

欠損補完: [1.0, 2.0, 3.0, 1000.0, 2.5]
正規化  : [0.0, 0.5, 1.0]


定数列  : [0.0, 0.0, 0.0] ← 0 除算しない


One-Hot : ([[0.0, 1.0], [1.0, 0.0], [0.0, 1.0]], [a, b])


## 3 つのモデルを同じ土俵で比較する

第 6 章のロジスティック回帰、第 9 章の決定木、第 12 章の AdaBoost を、第 7 章の指標で評価します。

**正解率だけを見ていると差を見落とします。** 指標ごとに順位が変わりうることに注目してください。

In [5]:
val evaluations = runPipeline(rows)

println("%-12s %8s %8s %8s %8s %8s".format("モデル", "正解率", "適合率", "再現率", "F1", "AUC"))
evaluations.forEach { e ->
    println("%-12s %8.3f %8.3f %8.3f %8.3f %8.3f".format(e.name, e.accuracy, e.precision,
            e.recall, e.f1, e.auc))
}

println()
println("F1 が最良のモデル: " + bestByF1(evaluations).name)

モデル               正解率      適合率      再現率       F1      AUC


logistic        0.750    1.000    0.250    0.400    1.000


tree            0.833    1.000    0.500    0.667    0.750


adaboost        0.917    1.000    0.750    0.857    1.000

F1 が最良のモデル: adaboost


## 試してみる: 評価は揺れる

テストデータは 12 件しかありません。**1 件の当たり外れが正解率を 0.083 動かします。**

分割のシードを変えて、どれくらい結果が揺れるか見てみましょう。「モデル A のほうが 3% 良い」という報告が、この規模では意味を持たないことが分かります。実務では交差検証で複数の分割を試し、平均と分散を見ます。

In [6]:
println("%6s %10s %8s".format("シード", "テスト件数", "陽性数"))
(0 until 4).forEach { seed ->
    val split = splitDataset(dataset, testRatio = 0.3, seed = seed)
    println("%6d %10d %8d".format(seed, split.testLabels.size, split.testLabels.sum()))
}

   シード      テスト件数      陽性数
     0         12        4


     1         12        5
     2         12        1
     3         12        3
